# Desafio 3 - ZettaLab

<img src="https://ufla.br/images/noticias/2020/04_abr/logo_zetta.png">

## Descrição do Problema

## Descrição da Solução

## Equipe

- Luciana Laibe Santos Silva, Comunicação e Marketing (9° Período, Graduação em Ciências Biológicas)
- Estevão Augusto da Fonseca Santos, Ciência e Governança de Dados (6° Período, Graduação em Ciência de Computação)
- Hugo Dias Pontello, Desenvolvimento de Software (5° Período, Graduação em Sistemas de Informação)
- Lorrana Verdi Flores, Desenvolvimento de Software (6° Período, Doutorado em Biotecnologia Vegetal)
- Bruna Oliveira Pereira, Geotecnologia (4° Período, Graduação em Engenharia Florestal)
- Geovanna Alexandre Possidonio, Gestão de Projetos (4° Período, Graduação em Administração)

## Importação de Bibliotecas

In [29]:
import pandas as pd             # Biblioteca para manipulação e análise de dados
import numpy as np              # Biblioteca para calculo de vetores e matrizes
import sys                      # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
import os                       # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
import basedosdados as bd       # Biblioteca para acessar o datalake público do site BasedosDados
import glob                     # Biblioteca para processamento em lote
from pathlib import Path        # Biblioteca para a manipulação de caminhos do sistema a nivel orientado a objetos
import gzip                     # Biblioteca para compressão e decompressão de dados usando o formato gzip
import shutil                   # Biblioteca para manipulação de arquivos e diretorios
import re

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import config_path          # Módulo que salva todos os caminhos de diretórios utilizados no projeto

from scripts import features        # Módulo de criação e manipulação de features
from scripts import modeling        # Módulo de modelagem de IA
from scripts import utils           # Módulo de utilitários genéricos
from scripts import pre_processing  # Módulo de pré-processamento e obtenção de dados.

from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env
load_dotenv()
GOOGLE_CLOUD_ID_PROJECT = os.getenv("GOOGLE_CLOUD_ID_PROJECT") # Coloca o ID do projeto do Google Cloud numa constante

## Obtenção e Tratamento de Dados

#### [Código de Múnicipios IBGE](https://www.ibge.gov.br/explica/codigos-dos-municipios.php)

A Tabela de Códigos de Municípios do IBGE apresenta a lista dos municípios brasileiros associados a um código composto de 7 dígitos, sendo os dois primeiros referentes ao código da Unidade da Federação.

É atualizada sistematicamente de forma a incluir as alterações decorrentes do desdobramento de municípios e, conseqüentemente, da criação de novos municípios, mudanças de nome dos municípios, como também de processos de fusão que resultam na extinção ou modificação de nome de algum município. 

In [2]:
url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/divisao_territorial/2024/DTB_2024.zip"

zip_file_path = utils.download_file(url, "codigo_municipios_ibge.zip", config_path.RAW_DATA_DIRECTORY_PATH)
utils.unzip_and_clean(zip_file_path, ['RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods'], config_path.RAW_DATA_DIRECTORY_PATH)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/codigo_municipios_ibge.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\Distritos_novos_e_extintos.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\Distritos_novos_e_extintos.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_DISTRITOS.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_DISTRITOS.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.xls
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_SUBDISTRITOS.ods
Removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\RELATORIO_DTB_BRASIL_2024_SUBDISTRITOS.xls
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\codigo_municipios_ibge.zip
Arquivos mantidos: [WindowsPath(

[WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods')]

In [4]:
df = pd.read_excel(f"{config_path.RAW_DATA_DIRECTORY_PATH}/RELATORIO_DTB_BRASIL_2024_MUNICIPIOS.ods")
df.drop('Unnamed: 9', axis=1, inplace=True)
df.columns = df.loc[5].to_list()
df.drop(index=5, inplace=True)
df.reset_index(drop=True, inplace=True)
df.rename({'UF' : 'ID_UF',
           'Código Município Completo' : 'ID_Município'
           }, axis=1, inplace=True)
df.dropna(axis=0, inplace=True) # Linhas

In [5]:
df

,ID_UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,ID_Município,Nome_Município
5,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,00015,1100015,Alta Floresta D'Oeste
6,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,00379,1100379,Alto Alegre dos Parecis
7,11,Rondônia,1101,Porto Velho,110002,Ariquemes,00403,1100403,Alto Paraíso
8,11,Rondônia,1102,Ji-Paraná,110004,Ji-Paraná,00346,1100346,Alvorada D'Oeste
9,11,Rondônia,1101,Porto Velho,110002,Ariquemes,00023,1100023,Ariquemes
...,...,...,...,...,...,...,...,...,...
5571,52,Goiás,5201,Goiânia,520002,Anápolis,22005,5222005,Vianópolis
5572,52,Goiás,5202,Itumbiara,520009,Piracanjuba,22054,5222054,Vicentinópolis
5573,52,Goiás,5206,Luziânia - Águas Lindas de Goiás,520022,Flores de Goiás,22203,5222203,Vila Boa
5574,52,Goiás,5205,Porangatu - Uruaçu,520018,Ceres - Rialma - Goianésia,22302,5222302,Vila Propício


In [6]:
df.drop(columns=[
    "Região Geográfica Intermediária", 
    "Nome Região Geográfica Intermediária",
    "Região Geográfica Imediata",
    "Nome Região Geográfica Imediata",
    "Município"
    ], axis=1, inplace=True)

df_uf = df[["ID_UF", "Nome_UF"]].drop_duplicates()
df_uf.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/codigos_ufs.csv", index=False)
df_mun = df[["ID_UF", "Nome_UF", "ID_Município", "Nome_Município"]]
df_mun.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/codigos_municipios.csv", index=False)

del df_uf
del df_mun

#### [Areas Territoriais - IBGE](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/estrutura-territorial/15761-areas-dos-municipios.html?t=acesso-ao-produto&c=1)

O redimensionamento dos valores de áreas é próprio da evolução das geotecnologias aplicadas no monitoramento da dinâmica da divisão territorial brasileira, que implica na atualização periódica dos valores das áreas estaduais e municipais com a utilização continuada de melhores técnicas e de melhores insumos de produção, além de refletir as eventuais alterações nos limites político-administrativos por justificativas legais ou judiciais.

In [7]:
url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/areas_territoriais/2024/AR_BR_RG_UF_RGINT_RGI_MUN_2024.ods"

file_name = utils.download_file(url, "area_territorial_brasil_regioes_ufs_muns.ods", config_path.RAW_DATA_DIRECTORY_PATH)
df_mun = pd.read_excel(file_name, "AR_BR_MUN_2024")
df_uf = pd.read_excel(file_name, "AR_BR_UF_2024")
df_rg = pd.read_excel(file_name, "AR_BR_RG_2024")

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/area_territorial_brasil_regioes_ufs_muns.ods


In [7]:
df_mun

,CD_UF,NM_UF,NM_UF_SIGLA,CD_MUN,NM_MUN,AR_MUN_2024
0,11,Rondônia,RO,1101203.0,Ministro Andreazza,798.083
1,11,Rondônia,RO,1101708.0,Urupá,831.857
2,11,Rondônia,RO,1100403.0,Alto Paraíso,2651.991
3,11,Rondônia,RO,1100452.0,Buritis,3265.810
4,11,Rondônia,RO,1101757.0,Vale do Anari,3135.106
...,...,...,...,...,...,...
5570,52,Goiás,GO,5213855.0,Morro Agudo de Goiás,282.333
5571,52,Goiás,GO,5201207.0,Anhanguera,55.569
5572,53,Distrito Federal,DF,5300108.0,Brasília,5760.783
5573,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_mun.dropna(how='any', axis=0, inplace=True)
df_mun.rename(columns={ 'CD_UF' : 'ID_UF',
                'NM_UF' : 'Nome_UF',
                'NM_UF_SIGLA' : 'UF_SIGLA',
                'CD_MUN' : 'ID_Município',
                'AR_MUN_2024' : 'AreaKm2_Mun_2024' 
                }, inplace=True)
df_mun.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/municipio_area_2024.csv", index=False)
df_mun

,ID_UF,Nome_UF,UF_SIGLA,ID_Município,NM_MUN,AreaKm2_Mun_2024
0,11,Rondônia,RO,1101203.0,Ministro Andreazza,798.083
1,11,Rondônia,RO,1101708.0,Urupá,831.857
2,11,Rondônia,RO,1100403.0,Alto Paraíso,2651.991
3,11,Rondônia,RO,1100452.0,Buritis,3265.810
4,11,Rondônia,RO,1101757.0,Vale do Anari,3135.106
...,...,...,...,...,...,...
5568,52,Goiás,GO,5202502.0,Aruanã,3054.773
5569,52,Goiás,GO,5210802.0,Itajá,2082.737
5570,52,Goiás,GO,5213855.0,Morro Agudo de Goiás,282.333
5571,52,Goiás,GO,5201207.0,Anhanguera,55.569


In [9]:
df_uf.dropna(how='any', axis=0, inplace=True)
df_uf.rename(columns={ 'CD_UF' : 'ID_UF',
               'NM_UF' : 'Nome_UF',
               'NM_UF_SIGLA' : 'UF_SIGLA',
               'AR_UF_2024' : 'AreaKm2_UF_2024'   
            }, inplace=True)
df_uf.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/uf_area_2024.csv", index=False)
df_uf

,ID_UF,Nome_UF,UF_SIGLA,AreaKm2_UF_2024
0,11,Rondônia,RO,237754.171
1,12,Acre,AC,164082.960
2,13,Amazonas,AM,1558706.127
3,14,Roraima,RR,223505.385
4,15,Pará,PA,1245828.829
5,16,Amapá,AP,142253.880
6,17,Tocantins,TO,277423.627
7,21,Maranhão,MA,329651.478
8,22,Piauí,PI,251755.499
9,23,Ceará,CE,148894.444


In [10]:
df_rg.dropna(how='any', axis=0, inplace=True)
df_rg.rename(columns={'CD_RG' : 'ID_RG',
             'NM_RG' : 'Nome_RG',
             'NM_RG_SIGLA' : 'RG_SIGLA',
             'AR_RG_2024' : 'AreaKm2_RG_2024'}, 
             inplace=True)
df_rg.to_csv(f"{config_path.PROCESSED_DATA_DIRECTORY_PATH}/rg_area_2024.csv", index=False)
df_rg

,ID_RG,Nome_RG,RG_SIGLA,AreaKm2_RG_2024
0,1,Norte,N,3849554.979
1,2,Nordeste,NE,1552175.419
2,3,Sudeste,SE,924558.342
3,4,Sul,S,576736.821
4,5,Centro-oeste,CO,1606354.015


In [11]:
del df_uf
del df_rg
del df_mun

#### BDQUEIMADAS - Dados

O BDQueimadas é o banco de dados de queimadas mantido pelo Instituto Nacional de Pesquisas Espaciais (INPE), que disponibiliza informações sobre focos de calor detectados por satélites. Esses focos indicam possíveis ocorrências de queimadas ou incêndios na vegetação, sendo monitorados continuamente em todo o território brasileiro e em outras regiões da América Latina.

Os dados fornecidos incluem informações como localização geográfica (latitude e longitude), data e hora da detecção, satélite responsável, além de atributos complementares como bioma, estado e intensidade do foco. Essas informações são coletadas por meio de sensoriamento remoto e organizadas em uma plataforma acessível ao público.

O BDQueimadas é amplamente utilizado para monitoramento ambiental, apoio à tomada de decisão por órgãos públicos, estudos científicos e análises de dados. A plataforma permite acesso aos dados por meio de downloads em formatos como CSV e GeoJSON, além de visualizações interativas e serviços que possibilitam integração com aplicações externas.

In [ ]:
df_mun['Nome_Município'] = df_mun['Nome_Município'].str.upper()
df_uf['Nome_UF'] = df_uf['Nome_UF'].str.upper()
df_mun = df_mun.loc[:, ['ID_Município','Nome_Município']]

##### BDQUEIMADAS - CSV - 2025 a 2022

In [30]:
def limpar_nome_arquivo(nome):
    # Substitui qualquer caractere que NÃO seja letras, números, pontos ou hífens por "_"
    # O Windows proíbe: < > : " / \ | ? *
    return re.sub(r'[^\w\d.-]', '_', nome)

urls = ["https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=ec9ed2ac-18c7-5e58-b05c-f2e5806f99eb&token=476fc698-b359-5459-b529-196ec5db488a&id=3143504f-a9d0-5dc1-af85-3a00041bdfa4",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=b0589a58-47a5-5377-9839-5904fc14afc9&token=476fc698-b359-5459-b529-196ec5db488a&id=824bad4d-3b7d-5ce3-97fd-ccee3d170a55",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=008d5526-dde7-508d-8cb3-3f30d74bf2e3&token=476fc698-b359-5459-b529-196ec5db488a&id=498ffaea-4d2d-558a-a556-d624a3027153",
       "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=c4a23f47-b03d-5611-bfd5-9081671d667d&token=476fc698-b359-5459-b529-196ec5db488a&id=b4ae70c1-e030-5c7a-86db-dbce2b6ce5df"]

nomes_zips = ["bdqueimadas_2025-01-01_2025-12-31.zip",
              "bdqueimadas_2024-01-01_2024-12-31.zip",
              "bdqueimadas_2023-01-01_2023-12-31.zip",
              "bdqueimadas_2022-01-01_2022-12-31.zip"]

nome_arquivos = ["bdqueimadas_2025-01-01_2025-12-31.csv",
                 "bdqueimadas_2024-01-01_2024-12-31.csv",
                 "bdqueimadas_2023-01-01_2023-12-31.csv",
                 "bdqueimadas_2022-01-01_2022-12-31.csv"]

for url, nome_arquivo, nome_zip in zip(urls, nome_arquivos, nomes_zips):
    zip_file_path = utils.download_file(url, nome_zip, config_path.RAW_DATA_DIRECTORY_PATH)
    file_content = utils.unzip_and_clean(zip_file_path, 'all', config_path.RAW_DATA_DIRECTORY_PATH)
    
    origem_suja = str(file_content[0])
    diretorio = os.path.dirname(origem_suja)
    nome_sujo = os.path.basename(origem_suja)
    nome_limpo = limpar_nome_arquivo(nome_sujo)
    
    caminho_corrigido = os.path.join(diretorio, nome_limpo)
    
    os.rename(caminho_corrigido,
              config_path.RAW_DATA_DIRECTORY_PATH / nome_arquivo)
    
    df = pd.read_csv(config_path.RAW_DATA_DIRECTORY_PATH / nome_arquivo)
    df = df.loc[df['Estado'] == "MINAS GERAIS"]
    df.dropna(how='any', axis=0, inplace=True)
    df.reset_index(inplace=True)

    df.rename(columns={
                'Estado' : 'Nome_UF',
                'Municipio' : 'Nome_Município'
            }, inplace=True)


    df_uf = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
    df_mun = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_municipios.csv")

    df = pd.merge(left=df, right=df_uf, how="inner", on=['Nome_UF'])
    df = pd.merge(left=df, right=df_mun, how="inner", on=['Nome_Município'])
    df.drop(columns='index', inplace=True)
    
    df.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / nome_arquivo, index=False)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2025-01-01_2025-12-31.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2025-01-01_2025-12-31.zip
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/bdqueimadas_2025-01-01_2025-12-31.csv')]
Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2024-01-01_2024-12-31.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2024-01-01_2024-12-31.zip
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/exportador_2025-09-25 20:02:36.991309.csv')]


FileNotFoundError: [WinError 2] O sistema não pode encontrar o arquivo especificado: 'd:\\projetos_github\\Desafio-3-ZettaLab-2ed\\data\\raw\\exportador_2025-09-25_20_02_36.991309.csv' -> 'd:\\projetos_github\\Desafio-3-ZettaLab-2ed\\data\\raw\\bdqueimadas_2024-01-01_2024-12-31.csv'

In [22]:
df

,DataHora,Satelite,Pais,Nome_UF,Nome_Município,Bioma,DiaSemChuva,Precipitacao,RiscoFogo,FRP,Latitude,Longitude,ID_UF,ID_Município
0,2025/01/01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.02,-999.00,1.3,-20.76235,-46.77194,31,3133758
1,2025/01/01 03:49:00,NOAA-20,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,1.01,-999.00,1.3,-20.76306,-46.77065,31,3133758
2,2025/01/01 04:44:00,NOAA-21,Brasil,MINAS GERAIS,ITAÚ DE MINAS,Mata Atlântica,0.0,0.98,-999.00,1.4,-20.76404,-46.76868,31,3133758
3,2025/01/01 15:56:00,NPP-375,Brasil,MINAS GERAIS,CARVALHOS,Mata Atlântica,0.0,1.29,0.00,3.7,-21.98483,-44.52124,31,3114808
4,2025/01/01 15:58:00,NPP-375,Brasil,MINAS GERAIS,COROMANDEL,Cerrado,0.0,2.83,0.00,0.8,-18.20023,-47.26058,31,3119302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207014,2025/12/31 19:50:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,122.7,-17.37550,-41.52950,31,3115458
207015,2025/12/31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,120.0,-17.37550,-41.52950,31,3115458
207016,2025/12/31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,139.5,-17.35530,-41.53460,31,3115458
207017,2025/12/31 20:00:00,GOES-19,Brasil,MINAS GERAIS,CATUJI,Mata Atlântica,10.0,0.00,0.11,132.2,-17.37620,-41.50410,31,3115458


In [ ]:
df.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "bdqueimadas_2025-01-01_2025-12-31_processado.csv", index=False)

##### BDQUEIMADAS - GeoJSON

In [2]:
urls = ["https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=da65dc59-0c98-5136-94bd-3070cffac964&token=476fc698-b359-5459-b529-196ec5db488a&id=59b5052c-d97e-5d3f-9960-89b7e0784666", # Amazônia
        "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=31f2c0c0-fe1a-590c-968f-d1460a8066e4&token=476fc698-b359-5459-b529-196ec5db488a&id=6fc7ee32-bfa6-5ecd-b930-b56f89a965f0", # Caatinga
        "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=302b48d7-6433-5cbb-99a2-b75a8a78e6f3&token=476fc698-b359-5459-b529-196ec5db488a&id=e7351481-e963-53b2-a63b-d0033f96b825", # Cerrado
        "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=dd415c7a-888c-5ffd-8e26-8525e6fb1e1b&token=476fc698-b359-5459-b529-196ec5db488a&id=a3599a44-9112-5b11-b0fb-f6f44e99e021", # Mata Atlântica
        "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=06a429e9-aba5-5562-85ec-d0ddb890ad08&token=476fc698-b359-5459-b529-196ec5db488a&id=124dfa86-4a12-540a-852a-2cdefc6f69ad", # Pampa
        "https://terrabrasilis.dpi.inpe.br/queimadas/download?arq=ba88257c-b909-58a1-9a7f-b1fee664be36&token=476fc698-b359-5459-b529-196ec5db488a&id=b7f80f67-bd48-5ecc-8c64-9a4b94294036"] # Pantanal

file_names = ["bdqueimadas_2025-01-01_2025-12-31_amazonia.geojson",
              "bdqueimadas_2025-01-01_2025-12-31_caatinga.geojson",
              "bdqueimadas_2025-01-01_2025-12-31_cerrado.geojson",
              "bdqueimadas_2025-01-01_2025-12-31_mata_atlantica.geojson",
              "bdqueimadas_2025-01-01_2025-12-31_pampa.geojson",
              "bdqueimadas_2025-01-01_2025-12-31_pantanal.geojson"]

for url, file in zip(urls, file_names):
    zip_file_path = utils.download_file(url, "bdqueimadas_2025-01-01_2025-12-31_geo_json-file.zip", config_path.RAW_DATA_DIRECTORY_PATH)
    utils.unzip_and_clean(zip_file_path, ['bdqueimadas_2025-01-01_2025-12-31.geojson'], config_path.RAW_DATA_DIRECTORY_PATH)
    os.rename(config_path.RAW_DATA_DIRECTORY_PATH / "bdqueimadas_2025-01-01_2025-12-31.geojson", 
              config_path.RAW_DATA_DIRECTORY_PATH / file)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2025-01-01_2025-12-31_geo_json-file.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2025-01-01_2025-12-31_geo_json-file.zip
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/bdqueimadas_2025-01-01_2025-12-31.geojson')]
Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2025-01-01_2025-12-31_geo_json-file.zip
Arquivos extraídos em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw
ZIP removido: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw\bdqueimadas_2025-01-01_2025-12-31_geo_json-file.zip
Arquivos mantidos: [WindowsPath('d:/projetos_github/Desafio-3-ZettaLab-2ed/data/raw/bdqueimadas_2025-01-01_2025-12-31.geojson')]
Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/bdqueimadas_2025-01

### [Banco de Dados Meteorológicos (BDMEP) - BaseDosDados](https://basedosdados.org/dataset/782c5607-9f69-4e12-b0d5-aa0f1a7a94e2?table=2c7fdc3d-f2ed-4c78-84b8-d9c792a06703)



O BDMEP abriga dados meteorológicos diários em forma digital, de séries históricas das várias estações meteorológicas convencionais da rede de estações do INMET com milhões de informações, referentes às medições diárias, de acordo com as normas técnicas internacionais da Organização Meteorológica Mundial.

**Organização**

Instituto Nacional de Meteorologia (INMET)

**Cobertura temporal**

2000 - 2025-12-31

In [ ]:
query = """
SELECT
    dados.ano as ano,
    dados.mes as mes,
    dados.data_hora as data_hora,
    dados.bioma as bioma,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.latitude as latitude,
    dados.longitude as longitude,
    dados.satelite as satelite,
    dados.dias_sem_chuva as dias_sem_chuva,
    dados.precipitacao as precipitacao,
    dados.risco_fogo as risco_fogo,
    dados.potencia_radiativa_fogo as potencia_radiativa_fogo
FROM `basedosdados.br_inpe_queimadas.microdados` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2020 AND ano <= 2025 AND sigla_uf = 'MG'
    """
caminho_completo = config_path.RAW_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP.parquet"
if not caminho_completo.exists():
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    df.to_parquet(caminho_completo, index=False)
    
df = pd.to_parquet(caminho_completo)
df

Downloading: 100%|██████████|


,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1715002,Nova Rosalândia,-10.50309,-49.01587,NOAA-21,0.0,24.09,0.00,1.9
1,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1717800,Ponte Alta do Bom Jesus,-11.88486,-46.53584,NOAA-21,2.0,2.24,0.03,1.5
2,2024,10,2024-10-30 04:21:00,Cerrado,MA,Maranhão,2102804,Carolina,-7.46682,-47.18346,NOAA-21,0.0,8.18,0.04,1.8
3,2024,10,2024-10-30 04:21:00,Cerrado,PI,Piauí,2201101,Avelino Lopes,-10.14042,-44.02159,NOAA-21,9.0,0.41,1.00,1.1
4,2024,10,2024-10-30 04:21:00,Caatinga,PE,Pernambuco,2612604,Santa Maria da Boa Vista,-8.74347,-39.68242,NOAA-21,12.0,0.00,1.00,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13510768,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.35668,-47.85893,NPP-375,2.0,0.50,0.04,6.0
13510769,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.15961,-47.89209,NPP-375,2.0,0.50,0.11,4.2
13510770,2024,12,2024-12-08 04:16:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.32073,-48.00707,NPP-375D,2.0,2.62,0.09,1.0
13510771,2024,12,2024-12-08 04:38:00,Cerrado,TO,Tocantins,1716208,Paranã,-13.08317,-48.11655,NOAA-20,0.0,1.00,0.05,1.3


In [6]:
df.dropna(how='any', axis=0, inplace=True)

In [7]:
df

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1715002,Nova Rosalândia,-10.50309,-49.01587,NOAA-21,0.0,24.09,0.00,1.9
1,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1717800,Ponte Alta do Bom Jesus,-11.88486,-46.53584,NOAA-21,2.0,2.24,0.03,1.5
2,2024,10,2024-10-30 04:21:00,Cerrado,MA,Maranhão,2102804,Carolina,-7.46682,-47.18346,NOAA-21,0.0,8.18,0.04,1.8
3,2024,10,2024-10-30 04:21:00,Cerrado,PI,Piauí,2201101,Avelino Lopes,-10.14042,-44.02159,NOAA-21,9.0,0.41,1.00,1.1
4,2024,10,2024-10-30 04:21:00,Caatinga,PE,Pernambuco,2612604,Santa Maria da Boa Vista,-8.74347,-39.68242,NOAA-21,12.0,0.00,1.00,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13510768,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.35668,-47.85893,NPP-375,2.0,0.50,0.04,6.0
13510769,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.15961,-47.89209,NPP-375,2.0,0.50,0.11,4.2
13510770,2024,12,2024-12-08 04:16:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.32073,-48.00707,NPP-375D,2.0,2.62,0.09,1.0
13510771,2024,12,2024-12-08 04:38:00,Cerrado,TO,Tocantins,1716208,Paranã,-13.08317,-48.11655,NOAA-20,0.0,1.00,0.05,1.3


In [8]:
df.to_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP_parquet.parquet")

In [9]:
del df

In [11]:
df = pd.read_parquet(config_path.PROCESSED_DATA_DIRECTORY_PATH / "dados_meteorologicos_BDMEP_parquet.parquet")

In [12]:
df

,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1715002,Nova Rosalândia,-10.50309,-49.01587,NOAA-21,0.0,24.09,0.00,1.9
1,2024,10,2024-10-30 04:21:00,Cerrado,TO,Tocantins,1717800,Ponte Alta do Bom Jesus,-11.88486,-46.53584,NOAA-21,2.0,2.24,0.03,1.5
2,2024,10,2024-10-30 04:21:00,Cerrado,MA,Maranhão,2102804,Carolina,-7.46682,-47.18346,NOAA-21,0.0,8.18,0.04,1.8
3,2024,10,2024-10-30 04:21:00,Cerrado,PI,Piauí,2201101,Avelino Lopes,-10.14042,-44.02159,NOAA-21,9.0,0.41,1.00,1.1
4,2024,10,2024-10-30 04:21:00,Caatinga,PE,Pernambuco,2612604,Santa Maria da Boa Vista,-8.74347,-39.68242,NOAA-21,12.0,0.00,1.00,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13510768,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.35668,-47.85893,NPP-375,2.0,0.50,0.04,6.0
13510769,2024,12,2024-12-07 17:09:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.15961,-47.89209,NPP-375,2.0,0.50,0.11,4.2
13510770,2024,12,2024-12-08 04:16:00,Cerrado,TO,Tocantins,1716208,Paranã,-12.32073,-48.00707,NPP-375D,2.0,2.62,0.09,1.0
13510771,2024,12,2024-12-08 04:38:00,Cerrado,TO,Tocantins,1716208,Paranã,-13.08317,-48.11655,NOAA-20,0.0,1.00,0.05,1.3


### INMET (Instituto Nacional de Meteorologia)

O Instituto Nacional de Meteorologia (INMET) é o órgão oficial federal do Brasil, vinculado ao Ministério da Agricultura e Pecuária, responsável pelo monitoramento, previsão do tempo e emissão de alertas meteorológicos. Criado em 1909, atua com uma rede de estações automáticas e convencionais para fornecer dados essenciais ao setor agropecuário e à população.

In [ ]:
url = "https://portal.inmet.gov.br/uploads/dadoshistoricos/2025.zip"

dir_caminho_inmet = config_path.RAW_DATA_DIRECTORY_PATH / "dataset_inmet"
dir_caminho_inmet_2025 = dir_caminho_inmet / "inmet_2025"
dir_caminho_inmet.mkdir(exist_ok=True, parents=True)
dir_caminho_inmet_2025.mkdir(exist_ok=True, parents=True)

zip_file_path = utils.download_file(url, "inmet_2025.zip", dir_caminho_inmet)
utils.unzip_and_clean(zip_file_path, 'all', dir_caminho_inmet_2025)

In [ ]:
arquivos = dir_caminho_inmet_2025.glob("*.CSV")
lista_df = []

# Colunas padronizadas que queremos manter
colunas_padrao = ['DATA', 'precip_mm', 'temp_max', 'temp_min', 'umid_max', 'umid_min', 'vento']

for arquivo in arquivos:
    try:
        # Ler as 8 primeiras linhas para pegar metadados
        with open(arquivo, encoding='latin1') as f:
            meta = [next(f).strip().split(';') for _ in range(8)]
            meta_dict = {linha[0].replace(":", "").strip(): linha[1].strip() for linha in meta}

        # Ler os dados, usando a 9ª linha como cabeçalho
        df = pd.read_csv(
            arquivo,
            sep=';',
            skiprows=8,
            decimal=',',
            encoding='latin1'
        )

        # Normalizar nomes de colunas
        df.columns = df.columns.str.strip().str.replace('\n','').str.replace('\r','')

        # Renomear colunas importantes
        df.rename(columns={
            'PRECIPITA��O TOTAL, HOR�RIO (mm)': 'precip_mm',
            'TEMPERATURA M�XIMA NA HORA ANT. (AUT) (�C)': 'temp_max',
            'TEMPERATURA M�NIMA NA HORA ANT. (AUT) (�C)': 'temp_min',
            'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)': 'umid_max',
            'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)': 'umid_min',
            'VENTO, VELOCIDADE HORARIA (m/s)': 'vento'
        }, inplace=True)

        # Criar coluna datetime
        df['DATA'] = pd.to_datetime(df['Data'] + ' ' + df['Hora UTC'], 
                                    format='%Y/%m/%d %H%M UTC', errors='coerce')

        # Adicionar informações da estação em cada linha
        df['ESTACAO'] = meta_dict.get('ESTACAO')
        df['UF'] = meta_dict.get('UF')
        df['LATITUDE'] = float(meta_dict.get('LATITUDE', 'nan').replace(',', '.'))
        df['LONGITUDE'] = float(meta_dict.get('LONGITUDE', 'nan').replace(',', '.'))
        df['CODIGO_WMO'] = meta_dict.get('CODIGO (WMO)')

        # Selecionar colunas padrão
        colunas_padrao = ['DATA', 'ESTACAO', 'UF', 'LATITUDE', 'LONGITUDE', 'CODIGO_WMO',
                          'precip_mm', 'temp_max', 'temp_min', 'umid_max', 'umid_min', 'vento']
        for col in colunas_padrao:
            if col not in df.columns:
                df[col] = pd.NA
        df = df[colunas_padrao]

        # Converter para numérico
        for col in ['precip_mm', 'temp_max', 'temp_min', 'umid_max', 'umid_min', 'vento', 'LATITUDE', 'LONGITUDE']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        lista_df.append(df)

    except Exception as e:
        print(f"Erro ao processar {arquivo}: {e}")

# Concatenar todos os arquivos
if lista_df:
    df_inmet_final = pd.concat(lista_df, ignore_index=True)
    print("Concatenação finalizada com sucesso!")
else:
    df_inmet_final = pd.DataFrame(columns=colunas_padrao)
    print("Nenhum arquivo válido para concatenar.")



# Salvar
df_inmet_final.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "df_inmet_2025.csv", index=False)
print(df_inmet_final.info())

Concatenação finalizada com sucesso!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9568080 entries, 0 to 9568079
Data columns (total 12 columns):
 #   Column      Dtype         
---  ------      -----         
 0   DATA        datetime64[ns]
 1   ESTACAO     object        
 2   UF          object        
 3   LATITUDE    float64       
 4   LONGITUDE   float64       
 5   CODIGO_WMO  object        
 6   precip_mm   float64       
 7   temp_max    float64       
 8   temp_min    float64       
 9   umid_max    float64       
 10  umid_min    float64       
 11  vento       float64       
dtypes: datetime64[ns](1), float64(8), object(3)
memory usage: 876.0+ MB
None


### [MapBiomas](https://brasil.mapbiomas.org/)

O MapBiomas é uma rede global e multi-institucional, formada por universidades, ONGs e empresas de tecnologia, que monitora as transformações na cobertura e no uso da terra nos territórios e seus impactos. Em 2025, a rede completou dez anos, fornecendo a mais atualizada e detalhada base de dados espaciais de uso da terra em um país disponível no mundo. Nascido no Brasil, o MapBiomas está atualmente presente em 14 países – toda a América do Sul e Indonésia.

Com base em ciência aberta e colaborativa, a rede alimenta uma plataforma que integra imagens de satélite, aprendizado de máquina e computação em nuvem. Todos os dados, mapas, métodos e códigos são disponibilizados de forma pública e gratuita.

As informações geradas pela rede MapBiomas podem ser utilizadas por tomadores de decisão, formuladores de políticas públicas, pesquisadores das mais diversas áreas, professores e estudantes, organizações da sociedade civil e empresas.

#### [MapBiomas - Fogo](https://plataforma.brasil.mapbiomas.org/coverage/coverage_lclu)

##### [MonitorFogo - MapBiomas](https://plataforma.monitorfogo.mapbiomas.org/)

In [14]:
df

,index,Ano,Bioma,Estados,Mês,mes,Nível 0,Nível 1,Nível 1_1,Nível 2,Nível 3,Nível 4,Área queimada (ha)
0,0,2019,Amazônia,Acre,Abril,4,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,11.043091
1,1,2019,Amazônia,Acre,Abril,4,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,0.177260
2,2,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Agricultura,Agricultura,Lavoura Temporária,Outras Lavouras Temporárias,46.909020
3,3,2019,Amazônia,Acre,Agosto,8,Antrópico,Agropecuária,Pastagem,Pastagem,Pastagem,Pastagem,66041.145360
4,4,2019,Amazônia,Acre,Agosto,8,Natural,Floresta,Formação Florestal,Floresta Alagável,Floresta Alagável,Floresta Alagável,68.431673
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25075,25075,2024,Pantanal,Mato Grosso do Sul,Março,3,Antrópico,Corpos D´água,Outros,"Rios, Lagos e Oceano","Rios, Lagos e Oceano","Rios, Lagos e Oceano",381.057779
25076,25076,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Florestal,Formação Florestal,Formação Florestal,Formação Florestal,433.061518
25077,25077,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Floresta,Formação Savânica,Formação Savânica,Formação Savânica,Formação Savânica,655.863783
25078,25078,2024,Pantanal,Mato Grosso do Sul,Março,3,Natural,Formação Natural não Florestal,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,Campo Alagado e Área Pantanosa,2370.983449


In [15]:
url = "https://docs.google.com/spreadsheets/d/1PZGPIrH1LIFj5Ob54yEVFZiRTOOuJaZ8pfKGZtFnobI/export?format=ods&gid=1141237655"

file_path = utils.download_file(url, "monitor-do-fogo-estatisticas.ods", config_path.RAW_DATA_DIRECTORY_PATH)
df = pd.read_excel(file_path, sheet_name='estatisticas')

df.to_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "monitor-fogo-mapbiomas.csv", 
          index=False)

Arquivo baixado e salvo em: d:\projetos_github\Desafio-3-ZettaLab-2ed\data\raw/monitor-do-fogo-estatisticas.ods


In [ ]:
del df

### [Banco de Dados de Queimadas - BaseDosDados](https://basedosdados.org/dataset/f06f3cdc-b539-409b-b311-1ff8878fb8d9?table=a3696dc2-4dd1-4f7e-9769-6aa16a1556b8)

Ocorrência do Fogo na Vegetação é o tema deste portal, http://queimadas.dgi.inpe.br/queimadas, desenvolvido no INPE, Instituto Nacional de Pesquisas Espaciais. Ele inclui o monitoramento operacional de focos de queimadas e de incêndios florestais detectados por satélites, e o cálculo e previsão do risco de fogo da vegetação.

Os dados para a América do Sul e a Central, África e Europa, são atualizados a cada três horas, todos os dias do ano. O acesso às informações é livre, e no navegador internet usado, deve estar desbloqueada a carga das janelas "pop up" para acesso a várias figuras, tabelas e gráficos.

Usuários que necessitam de focos com maior frequência temporal ou monitoramento em áreas específicas deverão consultar as "Perguntas Frequentes"

As informações neste portal estão divididas em blocos:

- "Situação Atual", é a "Sala de Situação" do Portal, e fornece para os últimos dois dias os resultados relevantes do monitoramento de focos de queima de vegetação feito pelo INPE em imagens satélites. Vários itens nos títulos das figuras e tabelas podem ser selecionados, alterando as apresentações gráficas conforme as preferências espaciais e temporais do usuário. Passando o mouse sobre os títulos e figuras serão indicadas as opções ativas.
- "SIG Focos-Geral", permite visualizar os focos em um Sistema de Informação Geográfica, com opções de períodos, regiões de interesse, satélites, planos de informação (p.ex. desmatamento, hidrografia, estradas), etc., além da exportação dos dados em formatos txt, html, shp e kmz.
- "SIG Focos-Áreas Protegidas", Semelhante ao item anterior, mas dedicado à ocorrência do fogo em Áreas de Preservação, como Parques, Florestas, Reservas Biológicas municipais, estaduais e nacionais, e Terras Indígenas.
- "Situação nas Áreas Protegidas", que contém o último relatório de focos detectados nas áreas de preservação, incluindo links que os mostram já inseridos no SIG do monitoramento.
- "Relatório Atual", contém o último resumo do monitoramento de queimadas em formato pdf, que poderá ser salvo pelo usuário para análise detalhada dos dados. A opção "Receber por E-Mail" leva ao cadastro do usuário, onde se define individualmente o conteúdo dos relatórios e mensagens de alerta que serão recebidos automaticamente por E-Mail.
- "Risco de Fogo", que apresenta o Risco de Fogo da vegetação estimado no presente, e suas previsões futuras (diárias, semanais e mensais), bem como o "fogograma" para qualquer local selecionado com o mouse nos mapas.
- "Meu Cadastro", onde o usuário define e configura individualmente os produtos que deseja receber em seu E-Mail, como alertas de ocorrência de focos em áreas de preservação, boletins pdf diários com tabelas, gráficos e mapas de estados e países de interesse, e mensagens operacionais.
- "Outros Produtos", com links a produtos adicionais gerados por este sistema de Queimadas e Incêndios, destacando-se: mapas mensais de focos em formato .gif com animações; mapas Meteorológicos (precipitação, temperatura, umidade do ar, ventos); detecções dos vários satélites utilizados; estimativas de concentrações e trajetórias dos poluentes emitidos; localização dos satélites utilizados, uma coletânea de dados anteriores e área de download, e; acesso a uma antiga versão da página do monitoramento de focos.
- "Links Relacionados", com indicações de acesso ao Boletim do Ibama de Monitoramento de Queimadas e Incêndios, ao texto publicado mensalmente no Boletim Climanálise, a uma coleção de mais de 500 páginas com matérias, fotos, vídeos e sites diversos sobre queimadas e incêndios florestais, à composição da equipe deste trabalho, e aos agradecimentos.
- "Notícias e Destaques", com informações, notícias e links de destaque na ocorrência do fogo na vegetação, e não restritas apenas à detecção de focos com satélites.
- "Aviso", com informações operacionais indicando novidades, falhas e detalhes relevantes na geração dos produtos do monitoramento do fogo na vegetação.

**Organização**

Instituto Nacional de Pesquisas Espaciais (INPE)

**Cobertura Temporal**

2000-2025

In [ ]:
query = """
SELECT
    dados.ano as ano,
    dados.mes as mes,
    dados.data_hora as data_hora,
    dados.bioma as bioma,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.latitude as latitude,
    dados.longitude as longitude,
    dados.satelite as satelite,
    dados.dias_sem_chuva as dias_sem_chuva,
    dados.precipitacao as precipitacao,
    dados.risco_fogo as risco_fogo,
    dados.potencia_radiativa_fogo as potencia_radiativa_fogo
FROM `basedosdados.br_inpe_queimadas.microdados` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2020 AND ano <= 2025 AND sigla_uf = 'MG'
    """
caminho_completo = config_path.RAW_DATA_DIRECTORY_PATH / "banco_dados_queimadas_raw.parquet"
if not caminho_completo.exists():
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    
    df_uf = pd.read_csv(config_path.PROCESSED_DATA_DIRECTORY_PATH / "codigos_ufs.csv")
    df.rename(columns={
                'sigla_uf' : 'Sigla_UF',
                'sigla_uf_nome' : 'Nome_UF',
                'id_municipio' : 'ID_Município',
                'id_municipio_nome' : 'Nome_Município'
                },
                inplace=True)
    df = pd.merge(left=df, right=df_uf, how="inner", on=["Nome_UF"])
    
    del df_uf
    df.to_parquet(caminho_completo, index=False)
    
df = pd.read_parquet(caminho_completo)
df

Downloading: 100%|██████████|


,ano,mes,data_hora,bioma,sigla_uf,sigla_uf_nome,id_municipio,id_municipio_nome,latitude,longitude,satelite,dias_sem_chuva,precipitacao,risco_fogo,potencia_radiativa_fogo
0,2024,8,2024-08-31 12:47:27,Mata Atlântica,SC,Santa Catarina,4205803,Garuva,-26.04640,-48.95320,GOES-16,-999.0,0.0,0.5,115.5
1,2024,8,2024-08-31 15:37:00,Mata Atlântica,ES,Espírito Santo,3201159,Brejetuba,-20.23871,-41.34359,NOAA-21,96.0,0.0,1.0,34.6
2,2024,8,2024-08-31 16:05:00,Caatinga,RN,Rio Grande do Norte,2407104,Macaíba,-6.02403,-35.50732,NPP-375,26.0,0.0,1.0,19.8
3,2024,8,2024-08-31 16:05:00,Caatinga,PE,Pernambuco,2612554,Santa Filomena,-8.33915,-40.53336,NPP-375,51.0,0.0,1.0,15.5
4,2024,8,2024-08-31 16:26:00,Mata Atlântica,ES,Espírito Santo,3201902,Domingos Martins,-20.31597,-41.08064,NOAA-20,21.0,0.0,1.0,14.2


In [ ]:
del df